# Phase 6: DNA Encoding Optimization

This notebook compares the existing char-based CUDA Hamming Distance implementation against the encoded `uint8_t` implementation. It assumes the current working directory is the repository root.

If the repository is not already available in Colab, clone it manually in a separate cell, then change into the repository directory. Do not run a clone command automatically if you are already in the project root.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
required_paths = [Path("src"), Path("scripts"), Path("benchmarks"), Path("README.md")]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(f"This notebook must be run from the repository root. Missing: {missing_paths}")

In [ ]:
!g++ src/dna_encoding.cpp -O3 -std=c++17 -I src/common -o dna_encoding

In [ ]:
!nvcc src/hamming_gpu.cu -O3 -std=c++17 -I src/common -o hamming_gpu

In [ ]:
!nvcc src/hamming_gpu_encoded.cu -O3 -std=c++17 -I src/common -o hamming_gpu_encoded

In [ ]:
!mkdir -p data/synthetic results/hamming benchmarks assets/benchmark_charts/encoding
!python scripts/generate_synthetic_dataset.py \
  --num-pairs 10000 \
  --sequence-length 128 \
  --output data/synthetic/synthetic_pairs_128.txt \
  --seed 42

In [ ]:
!./dna_encoding data/synthetic/synthetic_pairs_128.txt

In [ ]:
!./hamming_gpu \
  data/synthetic/synthetic_pairs_128.txt \
  results/hamming/hamming_gpu_char_results.csv \
  --repetitions 5

In [ ]:
!./hamming_gpu_encoded \
  data/synthetic/synthetic_pairs_128.txt \
  results/hamming/hamming_gpu_encoded_results.csv \
  --repetitions 5

In [ ]:
import filecmp

char_results = "results/hamming/hamming_gpu_char_results.csv"
encoded_results = "results/hamming/hamming_gpu_encoded_results.csv"
print("Char and encoded result files match:", filecmp.cmp(char_results, encoded_results, shallow=False))

In [ ]:
!python benchmarks/run_encoding_benchmark.py

In [ ]:
!python scripts/plot_encoding_benchmarks.py

In [ ]:
import pandas as pd

benchmark_path = "benchmarks/dna_encoding_benchmark_results.csv"
benchmark_results = pd.read_csv(benchmark_path)
display(benchmark_results.head())
print("Benchmark results saved to:", benchmark_path)

In [ ]:
from pathlib import Path

chart_directory = Path("assets/benchmark_charts/encoding")
for chart_path in sorted(chart_directory.glob("*.png")):
    print(chart_path)